# NYC 311 — Step 1: Download the real project dataset

This notebook downloads **real records directly from the official NYC Open Data API** and saves them as a CSV that we will later use in Python, Spark/Dataproc and (if useful) Power BI.

**Default:** complete-year filter for 2024, first **500,000** ordered service requests, selected project-relevant columns.

Output: `data/nyc311_2024_500k_raw.csv`


In [1]:
# Run this once if requests/pandas are missing
%pip install -q pandas requests


Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from io import StringIO
import time
import pandas as pd
import requests

API_URL = 'https://data.cityofnewyork.us/resource/erm2-nwe9.csv'
START_DATE = '2024-01-01T00:00:00'
END_DATE = '2025-01-01T00:00:00'
TARGET_ROWS = 500_000
PAGE_SIZE = 50_000

SELECT_FIELDS = [
    'unique_key', 'created_date', 'closed_date', 'agency', 'agency_name',
    'complaint_type', 'descriptor', 'location_type', 'incident_zip',
    'status', 'borough', 'latitude', 'longitude', 'open_data_channel_type'
]

WHERE = f"created_date >= '{START_DATE}' AND created_date < '{END_DATE}'"
OUTPUT_DIR = Path('data')
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / 'nyc311_2024_500k_raw.csv'

print('Output file:', OUTPUT_FILE)


Output file: data/nyc311_2024_500k_raw.csv


In [3]:
# Download in 50,000-row pages and append directly to CSV.
# This avoids holding all 500,000 rows in memory at once.
if OUTPUT_FILE.exists():
    OUTPUT_FILE.unlink()

session = requests.Session()
offset = 0
total = 0
page = 1

while total < TARGET_ROWS:
    requested = min(PAGE_SIZE, TARGET_ROWS - total)
    params = {
        '$select': ','.join(SELECT_FIELDS),
        '$where': WHERE,
        '$order': 'created_date ASC, unique_key ASC',
        '$limit': requested,
        '$offset': offset,
    }

    for attempt in range(1, 6):
        try:
            response = session.get(API_URL, params=params, timeout=180)
            response.raise_for_status()
            break
        except requests.RequestException:
            if attempt == 5:
                raise
            time.sleep(attempt * 3)

    chunk = pd.read_csv(StringIO(response.text), low_memory=False)
    if chunk.empty:
        break

    chunk.to_csv(
        OUTPUT_FILE,
        mode='w' if total == 0 else 'a',
        header=(total == 0),
        index=False
    )

    total += len(chunk)
    offset += len(chunk)
    print(f'Page {page:02d}: {len(chunk):,} rows | total {total:,}')
    page += 1
    if len(chunk) < requested:
        break
    time.sleep(0.4)

print('\nFinished.')
print(f'Rows saved: {total:,}')
print(f'File size: {OUTPUT_FILE.stat().st_size/(1024**2):.1f} MB')
print('Saved to:', OUTPUT_FILE.resolve())


Page 01: 50,000 rows | total 50,000
Page 02: 50,000 rows | total 100,000
Page 03: 50,000 rows | total 150,000
Page 04: 50,000 rows | total 200,000
Page 05: 50,000 rows | total 250,000
Page 06: 50,000 rows | total 300,000
Page 07: 50,000 rows | total 350,000
Page 08: 50,000 rows | total 400,000
Page 09: 50,000 rows | total 450,000
Page 10: 50,000 rows | total 500,000

Finished.
Rows saved: 500,000
File size: 102.4 MB
Saved to: /Users/leonard/Downloads/NYC311_Project_Starter/data/nyc311_2024_500k_raw.csv


In [5]:
# Verification: inspect the real headers and first rows.
preview = pd.read_csv(OUTPUT_FILE, nrows=10, low_memory=False)
print('Columns:', preview.columns.tolist())
display(preview)


Columns: ['unique_key', 'created_date', 'closed_date', 'agency', 'agency_name', 'complaint_type', 'descriptor', 'location_type', 'incident_zip', 'status', 'borough', 'latitude', 'longitude', 'open_data_channel_type']


,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,location_type,incident_zip,status,borough,latitude,longitude,open_data_channel_type
0,59894610,2024-01-01T00:00:00.000,2024-01-02T09:46:32.000,DOHMH,Department of Health and Mental Hygiene,Food Poisoning,1 or 2,Restaurant/Bar/Deli/Bakery,10304.0,Closed,STATEN ISLAND,40.628828,-74.074387,ONLINE
1,59896272,2024-01-01T00:00:00.000,2024-01-02T00:00:01.000,DOHMH,Department of Health and Mental Hygiene,Food Poisoning,1 or 2,Restaurant/Bar/Deli/Bakery,11211.0,Closed,BROOKLYN,40.710675,-73.954050,ONLINE
2,59898042,2024-01-01T00:00:00.000,2024-01-02T09:24:45.000,DOHMH,Department of Health and Mental Hygiene,Food Poisoning,1 or 2,Restaurant/Bar/Deli/Bakery,10462.0,Closed,BRONX,40.845340,-73.865594,ONLINE
3,59899838,2024-01-01T00:00:00.000,2024-01-02T09:42:25.000,DOHMH,Department of Health and Mental Hygiene,Food Poisoning,1 or 2,Restaurant/Bar/Deli/Bakery,10022.0,Closed,MANHATTAN,40.759192,-73.962290,ONLINE
4,59887043,2024-01-01T00:00:20.000,2024-01-01T00:41:01.000,NYPD,New York City Police Department,Illegal Fireworks,NaN,Street/Sidewalk,11228.0,Closed,BROOKLYN,40.627094,-74.014497,ONLINE
5,59889016,2024-01-01T00:00:20.000,2024-01-01T00:59:31.000,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,Residential Building/House,10468.0,Closed,BRONX,40.862406,-73.911097,PHONE
6,59888940,2024-01-01T00:00:43.000,2024-01-01T00:56:45.000,NYPD,New York City Police Department,Noise - Residential,Banging/Pounding,Residential Building/House,11207.0,Closed,BROOKLYN,40.658932,-73.884863,MOBILE
7,59891528,2024-01-01T00:00:46.000,2024-01-01T01:07:17.000,NYPD,New York City Police Department,Illegal Fireworks,NaN,Residential Building/House,11417.0,Closed,QUEENS,40.677497,-73.861919,ONLINE
8,59892651,2024-01-01T00:00:48.000,2024-01-01T00:48:34.000,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,Street/Sidewalk,11377.0,Closed,QUEENS,40.743761,-73.893186,ONLINE
9,59889054,2024-01-01T00:00:51.000,2024-01-01T00:29:32.000,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,Street/Sidewalk,11416.0,Closed,QUEENS,40.684738,-73.855753,MOBILE


## Stop here after the CSV is created

Do **not** clean or change the raw CSV yet. In the next stage we will make a baseline profile together: real row count, schema, missing values, date range, boroughs, complaint types and agencies. Then we will create a separate cleaned file.
